In [ ]:
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
import os
import json
import math
import numpy as np
import glob
import sys
sys.path.append("../")
# from src.evals.diversity_metrics import DiversityMetrics
from src.utils_v0 import list_to_str
from collections import Counter
from sklearn.metrics import cohen_kappa_score

In [ ]:
def load_sjts(hf_sjt_path):
    
    print("Using Huggingface SJTs")
    print(f"Loading SJTs from {hf_sjt_path}")
    hf_sjt_dataset = load_dataset(hf_sjt_path)
    sjt_datasets_total = hf_sjt_dataset['train']
    total_sjt_df = sjt_datasets_total.to_pandas()
    
    return total_sjt_df


def load_personas(hf_persona_path):
    
    print("Using Huggingface personas")
    print(f"Loading personas from {hf_persona_path}")
    hf_persona_dataset = load_dataset(hf_persona_path)
    persona_datasets_total = hf_persona_dataset['train']
    total_persona_df = persona_datasets_total.to_pandas()
    
    return total_persona_df


def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

def calculate_shannon_diversity(species_counts):
    """
    Calculates the Shannon diversity index (H) for a given set of species counts.

    Args:
        species_counts (list or dict): A list of counts for each species,
                                       or a dictionary where keys are species
                                       names and values are their counts.

    Returns:
        float: The Shannon diversity index (H).
    """

    total_individuals = sum(species_counts.values()) if isinstance(species_counts, dict) else sum(species_counts)
    
    if total_individuals == 0:
        return 0.0 # Handle case with no individuals

    shannon_index = 0.0
    for count in (species_counts.values() if isinstance(species_counts, dict) else species_counts):
        if count > 0:
            p_i = count / total_individuals
            shannon_index -= (p_i * math.log(p_i))
            
    return np.round(shannon_index,3).item()

def calculate_inverse_gini_index(distribution):
    
    proportions = [x / sum(distribution) for x in distribution]
    simpson_index = sum([p**2 for p in proportions])
    
    # The Gini-Simpson index is the inverse of Simpson's index D.
    # It represents the effective number of species.
    gini_simpson_index = 1 / simpson_index
    
    return np.round(gini_simpson_index,3).item()

In [ ]:
sjt_dataset = load_sjts("thoughtworks/psychometric_SJTs")

## Diversity Metrics for SJTs

In [ ]:
corrected_sjt_list = [sjt['corrected_sjt'] | {'hash_id': sjt['hash_id']} for sjt in sjt_dataset.to_dict("records")]
corrected_sjt_str_list = [" \n".join([sjt[key] for key in sjt.keys()if "hash_id" not in key]) for sjt in corrected_sjt_list]

In [ ]:
len(corrected_sjt_str_list)

In [ ]:
dm = DiversityMetrics(corrected_sjt_str_list, remove_stopwords=False)
print(dm.compute_all(k_for_silhouette=3))

## True Seeds Distribution

In [ ]:
config_list = [sjt['config'] | {'hash_id': sjt['hash_id']} for sjt in sjt_dataset.to_dict("records")]

In [ ]:
input_config_df = pd.DataFrame(config_list)

In [ ]:
input_config_df.columns

In [ ]:
cols_of_interest = ['age', 'ambiguity_level', 'authority_relationships',
       'gender', 'individuals_involved', 'race',
       'situation_type', 'threat_level', 'time_of_day', 'urgency_level']

for col in cols_of_interest:
    print(f"Distribution of '{col}':")
    print(input_config_df[col].value_counts()*100/input_config_df.shape[0])
    print("\n")

In [ ]:
calculate_shannon_diversity(input_config_df[col].value_counts().values)

In [ ]:
cols_of_interest = ['age', 'ambiguity_level', 'authority_relationships',
       'ethical_considerations', 'gender', 'individuals_involved', 'race',
       'situation_type', 'threat_level', 'time_of_day', 'urgency_level']
for col in cols_of_interest:
    print(f"Shannon Diversity Index for {col}")
    print(calculate_shannon_diversity(input_config_df[col].value_counts().values))
    print(f"Inverse Gini Index for {col}")
    print(calculate_inverse_gini_index(input_config_df[col].value_counts().values))

## Derived Seeds Distribution

In [ ]:
sjt_eval_total = {}
sjt_eval_dir = "../data/sjt_llm_judge_evaluation"

for filename in os.listdir(sjt_eval_dir):
    if "temp1point" in filename:
        print(filename)
        sjt_evals = read_json(os.path.join(sjt_eval_dir, filename))
        sjt_eval_total = sjt_eval_total | sjt_evals

In [ ]:
def get_derived_seeds(sjt_eval, key):

    rubric_2_eval = sjt_eval['sjt_rubric_2_evaluation']
    
    derived_seeds =  {key: rubric_2_eval[key]['value'] for key in rubric_2_eval if key!= "hexaco_traits" and key != 'rubric_quality'}

    derived_seeds['hash_id'] = key

    return derived_seeds

In [ ]:
get_derived_seeds(sjt_eval_total['ba8bb85f412fb49c6afa169644def0fcfdd9a1ee7d9abfe0c40cf259c6c5a4c0'],'ba8bb85f412fb49c6afa169644def0fcfdd9a1ee7d9abfe0c40cf259c6c5a4c0')

In [ ]:
derived_seeds_list = [get_derived_seeds(sjt_eval_total[key], key) for key in sjt_eval_total.keys()]

In [ ]:
derived_seeds_df = pd.DataFrame(derived_seeds_list)

In [ ]:
cols_of_interest = ['age', 'ambiguity_level', 'authority_relationships',
       'ethical_considerations', 'gender', 'individuals_involved', 'race',
       'situation_type', 'threat_level', 'time_of_day', 'urgency_level']

for col in cols_of_interest:
    print(f"Distribution of '{col}':")
    print(derived_seeds_df[col].value_counts()*100/derived_seeds_df.shape[0])
    print("\n")

In [ ]:
cols_of_interest = ['age', 'ambiguity_level', 'authority_relationships',
        'gender', 'individuals_involved', 'race',
       'situation_type', 'threat_level', 'time_of_day', 'urgency_level']
for col in cols_of_interest:
    print(f"Shannon Diversity Index for {col}")
    print(calculate_shannon_diversity(derived_seeds_df[col].value_counts().values))
    print(f"Inverse Gini Index for {col}")
    print(calculate_inverse_gini_index(derived_seeds_df[col].value_counts().values))

## Cohens Kappa between human annotators and LLM Judge

In [ ]:
def read_jsonl_file(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                json_object = json.loads(line.strip())  # Parse each line as JSON
                data.append(json_object)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON on line: {line.strip()}. Error: {e}")
    return data

In [ ]:
human_annotations_dir = "../data/sjt_data/sjt_human_annotations/*.jsonl"
human_annotations = []
keep_files = {
    "synthetic_generated_sjt_list_v8.1_temp_1point5.json",
}

# --- LOAD AND FILTER ---
for file_path in glob.glob(human_annotations_dir):
    with open(file_path, "r") as f:
        for line in f:
            data = json.loads(line)
            if data["file_name"] in keep_files:
                human_annotations.append(data)

# --- REMOVE DUPLICATES BASED ON hash_id ---
# Using pandas for convenience
human_annotations_df = pd.DataFrame(human_annotations)
human_annotations_df = human_annotations_df.drop_duplicates(subset=["hash_id"], keep="first")
human_annotations = human_annotations_df.to_dict("records")

In [ ]:
pd.Series([ann['file_name'] for ann in human_annotations]).value_counts()

In [ ]:
human_annotations_df['annotations'].loc[0]

In [ ]:
numeric_features = [
    'scenario_realism','ethical_tension', 'bias_fairness',
    'Honesty-Humility_alignment', 'Emotionality_alignment', 'Extraversion_alignment',
    'Agreeableness_alignment','Conscientiousness_alignment'
]

all_dict = {
    feature: [] for feature in numeric_features
}

In [ ]:
records = human_annotations_df['annotations'].to_list()
# records

In [ ]:
for record in records:
    for feature in numeric_features:
        all_dict[feature].append(int(record[feature]))

In [ ]:
import numpy as np
from Math import round

In [ ]:
all_dict
all_dict_averages = {item: round(np.average(scores), 2) for item, scores in all_dict.items()} 

In [ ]:
all_dict_averages

In [ ]:
llm_judge_annotations = {}
llm_judge_dir = "../data/sjt_llm_judge_evaluation"

for filename in os.listdir(llm_judge_dir):
    if "temp1point" in filename:
        print(filename)
        llm_judge_ann = read_json(os.path.join(llm_judge_dir, filename))
        llm_judge_annotations = llm_judge_annotations | llm_judge_ann

In [ ]:
keys_of_seeds = ['urgency_level', 'threat_level', 'ambiguity_level', 'individuals_involved', 'authority_relationships', 'situation_type', 'time_of_day', 'race', 'gender', 'age']

rubric_1_keys = ['scenario_realism', 'ethical_tension', 'bias_fairness']

In [ ]:
def preprocess_llm_judge_annotations(ann):
    return ann.replace("-","_").replace(" ","_").replace("/","_").\
        replace("native_american_or_alaska_native","native_american_alaska_native")

In [ ]:
seed_annotations_dict = {}
for human_ann in human_annotations:
    hash_id = human_ann['hash_id']
    
    llm_ann = llm_judge_annotations[hash_id]
    
    for key in keys_of_seeds:
        if key not in seed_annotations_dict.keys():
            seed_annotations_dict[key] = {"llm_judge":[preprocess_llm_judge_annotations(llm_ann['sjt_rubric_2_evaluation'][key]['value'].lower())],
                                    "human":[human_ann['annotations'][key].lower()]}
        else:
            seed_annotations_dict[key]['llm_judge'].append(preprocess_llm_judge_annotations(llm_ann['sjt_rubric_2_evaluation'][key]['value'].lower()))
            seed_annotations_dict[key]['human'].append(human_ann['annotations'][key].lower())

In [ ]:
rubric_1_annotations_dict = {}
for human_ann in human_annotations:
    hash_id = human_ann['hash_id']
    
    llm_ann = llm_judge_annotations[hash_id]
    for key in rubric_1_keys:
        if key == "bias_fairness":
                llm_judge_key = "fairness"
        else:
            llm_judge_key = key
        if key not in rubric_1_annotations_dict.keys():
            rubric_1_annotations_dict[key] = {"llm_judge":[preprocess_llm_judge_annotations(str(int(llm_ann['sjt_rubric_1_evaluation'][llm_judge_key]['score'])).lower())],
                                    "human":[human_ann['annotations'][key].lower()]}
        else:
            rubric_1_annotations_dict[key]['llm_judge'].append(preprocess_llm_judge_annotations(str(int(llm_ann['sjt_rubric_1_evaluation'][llm_judge_key]['score'])).lower()))
            rubric_1_annotations_dict[key]['human'].append(human_ann['annotations'][key].lower())
            

In [ ]:
human_ann['annotations']

In [ ]:
for human_ann in human_annotations:
    
    hexaco_ann = human_ann['annotations']
    
    hexaco_ann['honesty_humility_alignment'] = hexaco_ann.pop('honesty_humility_alignment')
    hexaco_ann['honesty_humility_overlap'] = hexaco_ann.pop('honesty_humility_overlap')
    
    hexaco_ann['emotionality_alignment'] = hexaco_ann.pop('emotionality_alignment')
    hexaco_ann['emotionality_overlap'] = hexaco_ann.pop('emotionality_overlap')
    
    hexaco_ann['extraversion_alignment'] = hexaco_ann.pop('extraversion_alignment')
    hexaco_ann['extraversion_overlap'] = hexaco_ann.pop('extraversion_overlap')
    
    hexaco_ann['agreeableness_alignment'] = hexaco_ann.pop('agreeableness_alignment')
    hexaco_ann['agreeableness_overlap'] = hexaco_ann.pop('agreeableness_overlap')
    
    hexaco_ann['conscientiousness_alignment'] = hexaco_ann.pop('conscientiousness_alignment')
    hexaco_ann['conscientiousness_overlap'] = hexaco_ann.pop('conscientiousness_overlap')
    
    hexaco_ann['openness_alignment'] = hexaco_ann.pop('openness_alignment')
    hexaco_ann['openness_overlap'] = hexaco_ann.pop('openness_overlap')

In [ ]:
hexaco_keys = ['honesty_humility','emotionality','extraversion','agreeableness','conscientiousness','openness']

In [ ]:
hexaco_annotations_dict = {}
for human_ann in human_annotations:
    hash_id = human_ann['hash_id']
    llm_ann = llm_judge_annotations[hash_id]
    for key in hexaco_keys:
        if f"{key}_alignment" not in hexaco_annotations_dict.keys():
            hexaco_annotations_dict[f"{key}_alignment"] = {"llm_judge":[preprocess_llm_judge_annotations(str(int(llm_ann['sjt_rubric_1_evaluation']['trait_alignment'][key]['score'])))],
                                    "human":[human_ann['annotations'][f"{key}_alignment"]]}
        else:
            hexaco_annotations_dict[f"{key}_alignment"]['llm_judge'].append(preprocess_llm_judge_annotations(str(int(llm_ann['sjt_rubric_1_evaluation']['trait_alignment'][key]['score']))))
            hexaco_annotations_dict[f"{key}_alignment"]['human'].append(human_ann['annotations'][f"{key}_alignment"])
            
        if f"{key}_overlap" not in hexaco_annotations_dict.keys():
            hexaco_annotations_dict[f"{key}_overlap"] = {"llm_judge":[preprocess_llm_judge_annotations(llm_ann['sjt_rubric_1_evaluation']['trait_alignment'][key]['overlaps'][0].lower() if llm_ann['sjt_rubric_1_evaluation']['trait_alignment'][key]['overlaps'] else "")],
                                    "human":["" if human_ann['annotations'][f"{key}_overlap"] == 'None' else human_ann['annotations'][f"{key}_overlap"].lower()]}
        else:
            hexaco_annotations_dict[f"{key}_overlap"]['llm_judge'].append(preprocess_llm_judge_annotations(llm_ann['sjt_rubric_1_evaluation']['trait_alignment'][key]['overlaps'][0].lower() if llm_ann['sjt_rubric_1_evaluation']['trait_alignment'][key]['overlaps'] else ""))
            hexaco_annotations_dict[f"{key}_overlap"]['human'].append("" if human_ann['annotations'][f"{key}_overlap"] == 'None' else human_ann['annotations'][f"{key}_overlap"].lower())
            

In [ ]:
complete_annotations_dict = seed_annotations_dict | rubric_1_annotations_dict | hexaco_annotations_dict

In [ ]:
complete_annotations_dict['ethical_tension']['human']

In [ ]:
cohens_kappa_dict = {}
for key in complete_annotations_dict.keys():
    
    llm_judge_ann = complete_annotations_dict[key]['llm_judge']
    human_ann = complete_annotations_dict[key]['human']
    kappa = np.round(cohen_kappa_score(llm_judge_ann, human_ann),4).item()
    if np.isnan(kappa):
        kappa = 1
    cohens_kappa_dict[key] = kappa

In [ ]:
np.nanmean(list(cohens_kappa_dict.values())).item()

In [ ]:
cohens_kappa_dict

## SJT Samples

In [ ]:
shortlisted_sjts = sjt_dataset[sjt_dataset['template_no'] == '4']

sampled_sjt = shortlisted_sjts.sample(1)
json.dumps(sampled_sjt['config'].values[0])

In [ ]:
sjt = sampled_sjt['corrected_sjt'].values[0]
" ".join([f"{key}: {sjt[key]}" for key in sjt.keys()])

In [ ]:
sampled_sjt['trait_bleed_evaluation'].values[0]

len(" ".join([f"{key}: {sampled_sjt['trait_bleed_evaluation'].values[0][key]}" for key in sampled_sjt['trait_bleed_evaluation'].values[0].keys()]).split())

## Persona Sample

In [ ]:
persona_dataset = load_personas("thoughtworks/psychometric_personas")

In [ ]:
persona_dataset['archetype'].unique()

In [ ]:
def categorize_age(age):
    if pd.isna(age):
        return "unknown"
    elif age < 18:
        return "juvenile"
    elif 18 <= age < 25:
        return "young_adult"
    elif 25 <= age < 40:
        return "adult"
    elif 40 <= age < 60:
        return "middle_aged"
    else:
        return "senior"

In [ ]:
persona_dataset['age_category'] = persona_dataset["age"].apply(categorize_age)

In [ ]:
persona_dataset['age_category'].value_counts()*100/persona_dataset.shape[0]

In [ ]:
persona_dataset['ethnic_background'].value_counts()*100/persona_dataset.shape[0]

In [ ]:
persona_dataset['sex'].value_counts()*100/persona_dataset.shape[0]

In [ ]:
shortlisted_personas = persona_dataset[persona_dataset['uuid'] == 'de216cca-84ff-4724-806a-ef96d530b450']

print(shortlisted_personas['archetype'])
sampled_persona = shortlisted_personas.sample(1)
print(sampled_persona['persona_string'].values[0])

## Seed Distributions

In [ ]:
seed_df = pd.json_normalize(sjt_dataset['config'])

## Persona Human Annotations

In [ ]:
persona_human_annotations_dir = "../data/persona_human_annotations/*.jsonl"
persona_human_annotations = []

# --- LOAD AND FILTER ---
for file_path in glob.glob(persona_human_annotations_dir):
    with open(file_path, "r") as f:
        for line in f:
            data = json.loads(line)
            persona_human_annotations.append(data)

# --- REMOVE DUPLICATES BASED ON hash_id ---
# Using pandas for convenience
persona_human_annotations_df = pd.DataFrame(persona_human_annotations)
# persona_human_annotations_df = persona_human_annotations_df.drop_duplicates(subset=["hash_id"], keep="first")
persona_human_annotations = persona_human_annotations_df.to_dict("records")

In [ ]:
persona_llm_judge = read_json("../data/persona_llm_judge/really_final_list_of_reviews.json")

In [ ]:
len([id for id in persona_human_annotations_df['persona_uuid'] if id in list(persona_llm_judge.keys())])

In [ ]:
persona_human_annotations_df['annotations'][0].keys()

In [ ]:
persona_annotation_keys = ['clarity', 'originality', 'coherence', 'diversity', 'realism', 'psychological_depth', 'consistency', 'informativeness', 'ethical_considerations', 'demographic_fidelity', 'overall_score']
persona_rubric_annotations_dict = {}
for human_ann in persona_human_annotations:
    uuid = human_ann['persona_uuid']
    llm_ann = persona_llm_judge[uuid]
    for key in persona_annotation_keys:
        if key not in persona_rubric_annotations_dict.keys():
            persona_rubric_annotations_dict[key] = {"llm_judge":[str(llm_ann[key]).lower()],
                                    "human":[str(human_ann['annotations'][key]).lower()]}
        else:
            persona_rubric_annotations_dict[key]['llm_judge'].append(str(llm_ann[key]).lower())
            persona_rubric_annotations_dict[key]['human'].append(str(human_ann['annotations'][key]).lower())
    

In [ ]:
persona_cohens_kappa_dict = {}
for key in persona_rubric_annotations_dict.keys():
    
    llm_judge_ann = persona_rubric_annotations_dict[key]['llm_judge']
    human_ann = persona_rubric_annotations_dict[key]['human']
    kappa = np.round(cohen_kappa_score(llm_judge_ann, human_ann),4).item()
    if np.isnan(kappa):
        kappa = 1
    persona_cohens_kappa_dict[key] = kappa

In [ ]:
persona_cohens_kappa_dict

In [ ]:
for key in persona_rubric_annotations_dict.keys():
    
    # llm_judge_ann = persona_rubric_annotations_dict[key]['llm_judge']
    print(key)
    print("Human: ",np.round(pd.Series(persona_rubric_annotations_dict[key]['human']).astype(int).mean(),3))
    print("LLM Judge: ",np.round(pd.Series(persona_rubric_annotations_dict[key]['llm_judge']).astype(int).mean(),3))
    